---
sidebar_label: Superlinked
---

# SIEEmbeddings

>[Superlinked](https://superlinked.com) is a self-hosted inference engine (SIE) for embedding, reranking, and extraction. The `sie-langchain` package provides LangChain components backed by a single SIE endpoint, giving you access to 85+ embedding models (dense, sparse, ColBERT/multivector).

This notebook shows how to get started with `SIEEmbeddings`. For reranking, use `SIEReranker` (a `BaseDocumentCompressor`), also included in the `sie-langchain` package.

## Setup

To use `SIEEmbeddings`, you need a running SIE instance and the `sie-langchain` package. See the [Superlinked quickstart](https://superlinked.com/docs) for deployment options (Docker, GPU, etc.).

### Installation

In [ ]:
%pip install --upgrade --quiet sie-langchain

This installs `sie-sdk` and `langchain-core` as dependencies.

### Start the SIE server

Run SIE locally with Docker:

```bash
docker run -p 8080:8080 ghcr.io/superlinked/sie-server:default
```

## Usage

`SIEEmbeddings` implements LangChain's `Embeddings` interface, so it drops into any vector store or retriever.

In [ ]:
from sie_langchain import SIEEmbeddings

embeddings = SIEEmbeddings(
    base_url="http://localhost:8080",
    model="BAAI/bge-m3",
)

# Embed documents
vectors = embeddings.embed_documents(
    [
        "Machine learning uses algorithms to learn from data.",
        "The weather is sunny today.",
    ]
)
print(len(vectors))

# Embed a query
query_vector = embeddings.embed_query("What is machine learning?")
print(len(query_vector))

Any of SIE's 85+ supported models works. Common picks:

| Model | Dims | Notes |
|---|---|---|
| `BAAI/bge-m3` | 1024 | Strong all-round, supports dense + sparse |
| `NovaSearch/stella_en_400M_v5` | 1024 | High retrieval quality |
| `nomic-ai/nomic-embed-text-v2-moe` | 768 | MoE, multilingual |
| `intfloat/e5-large-v2` | 1024 | Instruction-tuned, SIE handles query vs document encoding automatically |

See the [Model Catalog](https://superlinked.com/models) for all supported models.

### With a vector store

Use `SIEEmbeddings` as a drop-in embedding for any LangChain-supported vector store.

In [ ]:
from langchain_chroma import Chroma
from sie_langchain import SIEEmbeddings

embeddings = SIEEmbeddings(model="BAAI/bge-m3")

vectorstore = Chroma.from_texts(
    texts=[
        "Machine learning is a branch of artificial intelligence.",
        "Neural networks are inspired by biological neurons.",
        "Deep learning uses multiple layers of neural networks.",
    ],
    embedding=embeddings,
)

results = vectorstore.similarity_search("What is deep learning?", k=2)
for doc in results:
    print(doc.page_content)

### Async

Both sync and async methods are available:

```python
vectors = await embeddings.aembed_documents(texts)
query_vec = await embeddings.aembed_query(text)
```

## Reranking

The `sie-langchain` package also ships `SIEReranker`, which implements `BaseDocumentCompressor` and works with both cross-encoder models (e.g. `jinaai/jina-reranker-v2-base-multilingual`) and ColBERT / late-interaction models.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from sie_langchain import SIEReranker

reranker = SIEReranker(
    base_url="http://localhost:8080",
    model="jinaai/jina-reranker-v2-base-multilingual",
    top_k=5,
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 20}),
)

# Retrieves 20 docs, reranks with SIE, returns top 5
results = compression_retriever.invoke("What is machine learning?")

## API reference

- [`sie-langchain` on PyPI](https://pypi.org/project/sie-langchain/)
- [Superlinked on GitHub](https://github.com/superlinked/sie)
- [Superlinked docs](https://superlinked.com/docs)
- [Model Catalog](https://superlinked.com/models)